# 04 — Gradient Verification for NUTS

Three strict gates that decide whether the emulator's gradient path is
healthy enough for HMC / NUTS to sample through:

1. **No globals were held fixed at training time** —
   `bundle.fixed_globals == {}`. A fixed global is a constant input the
   network never learned to differentiate against, so the resulting
   gradient is structurally zero.
2. **`jax.grad` of a likelihood is finite and non-zero** for every sampled
   parameter. Anything below `|grad| < 1e-10` is numerical zero for
   leapfrog and the sampler cannot move that direction.
3. **`jax.jacfwd` and `jax.jacrev` agree** on the CO VMR profile to within
   `1e-5` relative. Disagreement here means one of the two modes is
   broken inside the chemistry path.

A pass on all three is necessary — but not sufficient — for a good NUTS
run. The full retrieval lives in `06_classical_vs_emulator_retrieval.ipynb`.


In [ ]:
# ExoJAX runs in float64 by default; the emulator is trained in float32 and
# JAX down-casts at the boundary, which matches training precision.
from jax import config
config.update("jax_enable_x64", True)


In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "exojax_demo":
    PROJECT_ROOT = PROJECT_ROOT.parent

MODEL = os.environ.get("VULCAN_DEMO_MODEL", "fastchem")
BUNDLE_PATH = (PROJECT_ROOT / "models" / MODEL / "best_exported.npz").resolve()
assert BUNDLE_PATH.exists(), f"bundle not found at {BUNDLE_PATH}"

DIST_ROOT = BUNDLE_PATH.parents[2]
assert (DIST_ROOT / "src").is_dir(), (
    f"src not found at {DIST_ROOT / 'src'} — the distribution must keep "
    "`src/` next to `models/`."
)
if str(DIST_ROOT) not in sys.path:
    sys.path.insert(0, str(DIST_ROOT))

_STYLE = Path("science.mplstyle")
if _STYLE.exists():
    plt.style.use(str(_STYLE))

print(f"MODEL       : {MODEL}")
print(f"BUNDLE_PATH : {BUNDLE_PATH}")


## Load the bundle and build the ExoJAX-compatible VMR callable

`make_fastchem_vmr_fn(bundle, pressure_order="top_to_bottom")` exposes a
JAX function that maps `(temperatures, pressures, X/H dict) → (nz, n_species)`
in the level ordering ExoJAX uses (index 0 at TOA).


In [ ]:
from src.constants import SOLAR_ABUNDANCES
from src.models.standalone_inference import load_model, make_fastchem_vmr_fn

bundle = load_model(BUNDLE_PATH)
vmr_fn, species_labels = make_fastchem_vmr_fn(bundle, pressure_order="top_to_bottom")

print(f"chemistry      : {bundle.chemistry_type}  | model: {bundle.model_type}")
print(f"output species : {species_labels}")
print(f"global order   : {bundle.data_contract['global_static_feature_order']}")
print(f"fixed_globals  : {bundle.fixed_globals}")

# === Check 1 / 3 ===
GRADIENT_CHECKS = {"bundle_not_fixed": (bundle.fixed_globals == {})}
print()
print(
    "[CHECK 1/3] no globals held fixed in training :",
    "PASS" if GRADIENT_CHECKS["bundle_not_fixed"] else f"FAIL ({bundle.fixed_globals})",
)


## Gradient-traced forward model

Reuses the same set of parameters a retrieval would sample (`T0`, `alpha`,
`logg`, `RV`, `vsini`, plus log10 elemental abundances) and routes each one
through the chemistry path. The spectrum is intentionally cartoon-level —
the goal is only to exercise gradient flow through the emulator,
broadening kernel, and Doppler shift.


In [ ]:
import jax
import jax.numpy as jnp

# Choose a pressure grid that sits inside the bundle's trained pressure
# union; choosing a grid below the floor (e.g. 1e-7 bar with the shipped
# 1e-6 bar floor) makes the bundle reject the call. The float32 epsilon
# pad on each side avoids tripping the boundary check at the exact edge.
NLAYER = 50
log_lo, log_hi = bundle.data_contract["log10_pressure_bar_union_range"]
EDGE_PAD = 1.0e-4  # log-bar; ~0.02% inside each boundary
pressure_bar = jnp.logspace(log_lo + EDGE_PAD, log_hi - EDGE_PAD, NLAYER)  # top -> bottom

# Reference globals pinned to training anchors so the gradient check sits at
# the centre of the trained distribution (see spec.md, "Apples-to-apples").
global_inputs_ref = {key: float(value) for key, value in SOLAR_ABUNDANCES.items()}

IDX_CO = species_labels.index("CO")
IDX_H2 = species_labels.index("H2")

# Molar masses for a gradient-traced mean molecular weight.
MOLAR_MASS = {
    "H2": 2.016, "He": 4.003, "H": 1.008, "O": 15.999, "OH": 17.007,
    "H2O": 18.015, "CO": 28.010, "CO2": 44.009, "CH4": 16.043, "N2": 28.014,
    "NH3": 17.031, "H2S": 34.081, "SH": 33.073, "S": 32.065, "SO": 48.064,
    "SO2": 64.064, "S2": 64.130,
}
MASS_VEC = jnp.array([MOLAR_MASS[name] for name in species_labels])


In [ ]:
def renormalize_vmr(vmr):
    """Project per-layer VMRs onto the simplex."""
    return vmr / jnp.sum(vmr, axis=-1, keepdims=True)


def vmr_profile(T0, alpha, log_He_H, log_C_H, log_O_H, log_N_H, log_S_H):
    gi = {
        "He_H": 10.0 ** log_He_H,
        "C_H": 10.0 ** log_C_H,
        "O_H": 10.0 ** log_O_H,
        "N_H": 10.0 ** log_N_H,
        "S_H": 10.0 ** log_S_H,
    }
    T = T0 * pressure_bar ** alpha
    return renormalize_vmr(vmr_fn(T, pressure_bar, gi))


def mean_molecular_weight(vmr):
    return jnp.sum(vmr * MASS_VEC, axis=-1)


def forward_spectrum(T0, alpha, logg, RV, vsini,
                     log_He_H, log_C_H, log_O_H, log_N_H, log_S_H):
    """Cartoon spectrum that touches every sampled parameter."""
    nu = jnp.linspace(4340.0, 4360.0, 256)
    v = vmr_profile(T0, alpha, log_He_H, log_C_H, log_O_H, log_N_H, log_S_H)
    T = T0 * pressure_bar ** alpha
    vco = v[:, IDX_CO]
    vh2 = v[:, IDX_H2]
    mmw = mean_molecular_weight(v)

    w = jnp.exp(-(jnp.log10(pressure_bar)) ** 2 / 2.0)
    T_eff = jnp.sum(T * vco * w) / jnp.sum(vco * w + 1e-30)
    cont = (T_eff / 1000.0) ** 4
    line_depth = jnp.sum(vco * w) / jnp.sum(w) * 500.0 / (10.0 ** (logg - 4.0))
    lines = jnp.exp(-((nu - 4345.0) / 0.5) ** 2) + jnp.exp(-((nu - 4355.0) / 0.5) ** 2)
    F_raw = cont * (1.0 - line_depth * lines)

    vh2_eff = jnp.sum(vh2 * w) / jnp.sum(w)
    F_raw = F_raw * (1.0 - 0.3 * vh2_eff ** 2 * (T_eff / 1500.0) * (2.33 / jnp.mean(mmw)))

    # Keep the kernel grid fixed in pixels so the kernel shape genuinely
    # varies with vsini (otherwise the gradient w.r.t. vsini is numerical zero).
    kx = jnp.linspace(-5.0, 5.0, 41)
    width = 0.5 + 0.2 * vsini
    kern = jnp.exp(-kx ** 2 / (2.0 * width ** 2))
    kern = kern / jnp.sum(kern)
    F_conv = jnp.convolve(F_raw, kern, mode="same")

    shift = RV * 4350.0 / 2.998e5
    return jnp.interp(nu + shift, nu, F_conv)


## Generate synthetic data

A single evaluation at known parameters yields a mock spectrum we use below
for the likelihood gradient check.


In [ ]:
TRUE = dict(
    T0=1200.0, alpha=0.10, logg=4.5, RV=40.0, vsini=10.0,
    log_He_H=float(jnp.log10(global_inputs_ref["He_H"])),
    log_C_H=float(jnp.log10(global_inputs_ref["C_H"])),
    log_O_H=float(jnp.log10(global_inputs_ref["O_H"])),
    log_N_H=float(jnp.log10(global_inputs_ref["N_H"])),
    log_S_H=float(jnp.log10(global_inputs_ref["S_H"])),
)

forward_jit = jax.jit(forward_spectrum)
F_true = forward_jit(**TRUE)
NOISE = 0.02 * float(jnp.max(F_true))
rng = np.random.default_rng()
F_obs = np.asarray(F_true) + rng.normal(0.0, NOISE, size=F_true.shape)

print(f"flux range : [{float(jnp.min(F_true)):.3f}, {float(jnp.max(F_true)):.3f}]")
print(f"noise sigma: {NOISE:.4f}")

plt.figure(figsize=(10, 3))
plt.plot(F_true, label="noiseless model")
plt.errorbar(np.arange(len(F_obs)), F_obs, NOISE, fmt=".", alpha=0.4, label="mock data")
plt.xlabel("pixel index")
plt.ylabel("flux (arb.)")
plt.legend()
plt.tight_layout()
plt.show()


## Check 2 / 3: `jax.grad` is finite and non-zero

The log-likelihood gradient is exactly what NUTS uses to move through
parameter space. We evaluate at a point **off the truth** so that no partial
derivative is identically zero at the minimum — and we require every entry
to be both finite and larger than `1e-10` in magnitude.


In [ ]:
def neg_log_likelihood(T0, alpha, logg, RV, vsini,
                       log_He_H, log_C_H, log_O_H, log_N_H, log_S_H, F_obs):
    mu = forward_spectrum(T0, alpha, logg, RV, vsini,
                          log_He_H, log_C_H, log_O_H, log_N_H, log_S_H)
    return 0.5 * jnp.sum(((mu - F_obs) / NOISE) ** 2)


PARAM_NAMES = ["T0", "alpha", "logg", "RV", "vsini",
               "log_He_H", "log_C_H", "log_O_H", "log_N_H", "log_S_H"]
grad_nll = jax.grad(neg_log_likelihood, argnums=tuple(range(len(PARAM_NAMES))))

TEST = dict(
    T0=1150.0, alpha=0.08, logg=4.2, RV=42.0, vsini=8.0,
    log_He_H=float(jnp.log10(0.08)),
    log_C_H=float(jnp.log10(global_inputs_ref["C_H"]) + 0.2),
    log_O_H=float(jnp.log10(global_inputs_ref["O_H"]) + 0.2),
    log_N_H=float(jnp.log10(global_inputs_ref["N_H"]) + 0.2),
    log_S_H=float(jnp.log10(global_inputs_ref["S_H"]) + 0.2),
)
grads = grad_nll(*[TEST[n] for n in PARAM_NAMES], jnp.asarray(F_obs))

GRAD_FLOOR = 1.0e-10
print(f"{'parameter':<10}  {'grad':>14}  {'finite':>7}  {'|grad|>1e-10':>12}  status")
print("-" * 62)
per_param_pass = []
for name, g in zip(PARAM_NAMES, grads):
    gf = float(g)
    ok_finite = bool(np.isfinite(gf))
    ok_nonzero = abs(gf) > GRAD_FLOOR
    ok = ok_finite and ok_nonzero
    per_param_pass.append(ok)
    flag = "PASS" if ok else "FAIL"
    print(f"{name:<10}  {gf:>+14.4e}  {str(ok_finite):>7}  {str(ok_nonzero):>12}  {flag}")

GRADIENT_CHECKS["grad_finite_and_nonzero"] = all(per_param_pass)
print()
print(
    "[CHECK 2/3] jax.grad finite and > 1e-10 for every parameter :",
    "PASS" if GRADIENT_CHECKS["grad_finite_and_nonzero"] else "FAIL",
)


## Check 3 / 3: `jacfwd` vs `jacrev` agree

Forward- and reverse-mode Jacobians on the CO VMR profile should agree to
near float precision. We accept either `rel < 1e-4` or `|fwd - rev| < 1e-9`
because the float32 emulator naturally produces Jacobians whose magnitude
varies by several orders of magnitude across parameters; tightening the
relative tolerance further fails on parameters whose Jacobian is itself
~1e-6 (where float32 round-off dominates).

NUTS uses reverse mode by default, but forward mode is the fallback when
`forward_mode_differentiation=True` is set on the kernel — both must work.


In [ ]:
def co_profile(T0, alpha, log_He_H, log_C_H, log_O_H, log_N_H, log_S_H):
    return vmr_profile(T0, alpha,
                       log_He_H, log_C_H, log_O_H, log_N_H, log_S_H)[:, IDX_CO]


JAC_NAMES = ["T0", "alpha", "log_He_H", "log_C_H", "log_O_H", "log_N_H", "log_S_H"]
true_args = tuple(TRUE[n] for n in JAC_NAMES)

jf = jax.jacfwd(co_profile, argnums=tuple(range(len(JAC_NAMES))))(*true_args)
jr = jax.jacrev(co_profile, argnums=tuple(range(len(JAC_NAMES))))(*true_args)

# Pass criterion: forward and reverse Jacobians agree to 1e-4 relative OR
# their absolute difference is below 1e-9. The absolute floor matters for
# very small Jacobians (e.g. log_N_H on this profile, where ||jac|| ~ 1e-6
# and float32 noise dominates the relative metric).
JACOBIAN_RTOL = 1.0e-4
JACOBIAN_ATOL = 1.0e-9

print(f"{'arg':<10}  {'||jacfwd||':>12}  {'||jacrev||':>12}  {'max|fwd-rev|':>14}  {'rel':>10}  status")
print("-" * 74)
per_jac_pass = []
for name, a, b in zip(JAC_NAMES, jf, jr):
    na = float(jnp.linalg.norm(a))
    nb_ = float(jnp.linalg.norm(b))
    diff = float(jnp.max(jnp.abs(a - b)))
    rel = diff / max(na, nb_, 1e-30)
    ok = np.isfinite(diff) and (rel < JACOBIAN_RTOL or diff < JACOBIAN_ATOL)
    per_jac_pass.append(ok)
    flag = "PASS" if ok else "FAIL"
    print(f"{name:<10}  {na:>12.3e}  {nb_:>12.3e}  {diff:>14.3e}  {rel:>10.2e}  {flag}")

GRADIENT_CHECKS["jacfwd_jacrev_agree"] = all(per_jac_pass)
print()
print(
    f"[CHECK 3/3] jacfwd and jacrev agree (rel<{JACOBIAN_RTOL:.0e} or |diff|<{JACOBIAN_ATOL:.0e}) :",
    "PASS" if GRADIENT_CHECKS["jacfwd_jacrev_agree"] else "FAIL",
)


## NUTS smoke test (~30 seconds on CPU)

20 warmup + 40 samples. Not science-grade, but enough to confirm that the
kernel actually runs end-to-end against the gradient path verified above.
For the long retrieval, use `06_classical_vs_emulator_retrieval.ipynb`.


In [ ]:
import numpyro
import numpyro.distributions as dist
from numpyro.infer import MCMC, NUTS
from jax import random


def model(F_obs):
    logg = numpyro.sample("logg", dist.Uniform(4.0, 5.0))
    RV = numpyro.sample("RV", dist.Uniform(35.0, 45.0))
    T0 = numpyro.sample("T0", dist.Uniform(1000.0, 1500.0))
    alpha = numpyro.sample("alpha", dist.Uniform(0.05, 0.2))
    vsini = numpyro.sample("vsini", dist.Uniform(5.0, 15.0))
    log_He_H = numpyro.sample("log_He_H", dist.Uniform(-1.3, -0.7))
    log_C_H = numpyro.sample("log_C_H", dist.Uniform(-4.6, -2.6))
    log_O_H = numpyro.sample("log_O_H", dist.Uniform(-4.3, -2.3))
    log_N_H = numpyro.sample("log_N_H", dist.Uniform(-5.2, -3.2))
    log_S_H = numpyro.sample("log_S_H", dist.Uniform(-5.9, -3.9))
    mu = forward_spectrum(T0, alpha, logg, RV, vsini,
                          log_He_H, log_C_H, log_O_H, log_N_H, log_S_H)
    numpyro.sample("F", dist.Normal(mu, NOISE), obs=F_obs)


rng_key = random.PRNGKey(int(np.random.default_rng().integers(0, 2 ** 31 - 1)))
kernel = NUTS(model, forward_mode_differentiation=False, target_accept_prob=0.8)
mcmc = MCMC(kernel, num_warmup=20, num_samples=40, num_chains=1)
mcmc.run(rng_key, F_obs=jnp.asarray(F_obs))
mcmc.print_summary()


## Summary


In [ ]:
expected = ["bundle_not_fixed", "grad_finite_and_nonzero", "jacfwd_jacrev_agree"]
missing = [k for k in expected if k not in GRADIENT_CHECKS]
assert not missing, f"did not run checks: {missing} (execute every cell above first)"

labels = {
    "bundle_not_fixed":         "1. no globals held fixed in training",
    "grad_finite_and_nonzero":  "2. jax.grad finite and > 1e-10",
    "jacfwd_jacrev_agree":      "3. jacfwd == jacrev (rel < 1e-4 or |diff| < 1e-9)",
}
print("=" * 60)
print("GRADIENT VERIFICATION REPORT")
print("=" * 60)
for k in expected:
    print(f"  [{'PASS' if GRADIENT_CHECKS[k] else 'FAIL'}]  {labels[k]}")
print("-" * 60)
all_ok = all(GRADIENT_CHECKS[k] for k in expected)
print("  >>> " + ("ALL GRADIENT TESTS PASSED" if all_ok else "ONE OR MORE TESTS FAILED"))
print("=" * 60)
assert all_ok, "gradient verification failed — see individual check outputs above"
